# Replacing Fiji with Python

The pipeline depends on Fiji for two things: reading Olympus `.oir` through
Bio-Formats, and pixel classification in Labkit. Labkit is a Java GUI plugin, so
any tool built on it requires a user to install Fiji, open a window, and run a
macro. That is the single largest obstacle to anyone else using this.

This notebook works out how much of it can be replaced, and measures whether the
replacement gives the same answer.

**The result, up front:** the Python reimplementation agrees with Labkit on a
median 93% of segmented pixels, and the fragmentation index computed from its
masks correlates 0.990 with the index computed from Labkit's. Everything except
reading the vendor file format ports cleanly.

---

## 0. Setup

In [ ]:
import importlib, subprocess, sys
for mod, pkg in [('skimage', 'scikit-image'), ('tifffile', 'tifffile')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import json, os, urllib.request, zipfile
import numpy as np, matplotlib as mpl, matplotlib.pyplot as plt, tifffile
from scipy import ndimage as ndi
from skimage.feature import hessian_matrix, hessian_matrix_eigvals

REPO = 'https://raw.githubusercontent.com/vishal9045/mitofi/master'
if not os.path.isdir('port_example'):
    urllib.request.urlretrieve(f'{REPO}/data/port_example.zip', 'port_example.zip')
    zipfile.ZipFile('port_example.zip').extractall('.')
print(sorted(os.listdir('port_example')))

In [ ]:
C = {'blue':'#2a78d6','orange':'#eb6834','aqua':'#1baf7a',
     'ink':'#0b0b0b','ink2':'#52514e','muted':'#8a8983','grid':'#e6e5e1','surface':'#fcfcfb'}
mpl.rcParams.update({
    'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'],
    'savefig.facecolor': C['surface'], 'axes.edgecolor': C['grid'], 'axes.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': C['ink2'], 'axes.titlecolor': C['ink'],
    'axes.titlesize': 12, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.titlepad': 12, 'axes.labelsize': 10,
    'xtick.color': C['ink2'], 'ytick.color': C['ink2'],
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'grid.color': C['grid'],
    'legend.frameon': False, 'legend.fontsize': 9,
    'font.size': 10, 'lines.linewidth': 2, 'figure.dpi': 120})

---
## 1. What actually depends on Fiji

| Step | Why it is on Fiji | Python route | Verdict |
|---|---|---|---|
| Read `.oir` | Bio-Formats | Bio-Formats **headless** via `scyjava`, or `bioio` | Java stays, GUI goes |
| Read `.lif` (Leica) | Bio-Formats | `readlif` - pure Python | Java goes entirely |
| Pick channel, project | inside the macro | `numpy.max` over Z | trivial |
| Build training montage | inside the macro | array paste | trivial |
| **Pixel classification** | Labkit GUI | this notebook | the real question |
| Draw *new* scribbles | Labkit GUI | napari labels layer | training only |

Only the last two matter. Everything above them is plumbing.

Note the asymmetry worth knowing about: your Olympus files need Java to read,
but Leica `.lif` has a pure-Python reader. The planned second dataset is the
easier one.

---
## 2. The Labkit classifier is mostly open

A `.classifier` file looks opaque. It is not - it is JSON.

In [ ]:
spec = json.load(open('port_example/classifier_v5_spec.json'))
print(json.dumps(spec['features']['globals'], indent=1))
print('classes:', spec['classNames'])
print()
for op in spec['features']['ops']:
    print(' ', op['class'].rsplit('.', 1)[-1])

Six filter families at four scales, on a two-class problem. Every one of those
filters exists in scipy or scikit-image.

The one part that does **not** port is the trained forest itself. In the full
file it is a base64 blob beginning `rO0ABX`, which is Java object serialisation
(`\xac\xed` magic) wrapping a Weka `FastRandomForest`. Parsing that into
scikit-learn is possible and not worth it.

It does not need to port, because the thing it was trained *on* is also open.

---
## 3. The scribbles are open too

Labkit's `.labeling` file is JSON: the literal pixel coordinates that were
painted, per class.

In [ ]:
lab = json.load(open('port_example/montage.tif.labeling'))
print('interval :', lab['interval'])
print('colours  :', lab['colors'])
for k, v in lab['labels'].items():
    print(f'  {k:<14} {len(v):>7} pixels   first: {v[0]}')

scribbles = {k: np.array(v, dtype=int) for k, v in lab['labels'].items()}

In [ ]:
montage = tifffile.imread('port_example/montage.tif')
print('montage', montage.shape, montage.dtype)

y0, y1, x0, x1 = 900, 1500, 900, 1500
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
for a in ax:
    a.imshow(montage[y0:y1, x0:x1], cmap='gray',
             vmax=np.percentile(montage[y0:y1, x0:x1], 99.5))
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
ax[0].set_title('training montage (detail)')
for name, col in [('background', C['blue']), ('Mitochondria', C['orange'])]:
    s = scribbles[name]
    inside = s[(s[:,0]>=x0)&(s[:,0]<x1)&(s[:,1]>=y0)&(s[:,1]<y1)]
    ax[1].scatter(inside[:,0]-x0, inside[:,1]-y0, s=1.2, color=col, label=name)
ax[1].set_title('the scribbles, recovered from JSON')
ax[1].legend(markerscale=8, loc='lower right')
plt.tight_layout(); plt.show()

---
## 4. Rebuilding the feature stack

25 planes: the raw image, plus six filters at sigma 1, 2, 4 and 8 pixels.

One caveat stated rather than buried. Labkit's JSON records *which* filters and
*which* sigmas, but not the sigma ratio its difference-of-Gaussians uses. The
conventional half-sigma pairing is used here. That makes this an equivalent
stack, not a bit-exact reproduction - which is precisely why section 6 measures
the disagreement instead of assuming there is none.

In [ ]:
def feature_stack(img, sigmas):
    """Yield (name, plane) one at a time - materialising 25 float32 planes of a
    3072-square montage at once costs about a gigabyte for no reason."""
    im = img.astype(np.float32)
    yield 'identity', im
    for s in sigmas:
        g = ndi.gaussian_filter(im, s)
        yield f'gauss{s:g}',   g
        yield f'dog{s:g}',     g - ndi.gaussian_filter(im, s / 2)
        yield f'gradmag{s:g}', ndi.gaussian_gradient_magnitude(im, s)
        yield f'log{s:g}',     ndi.gaussian_laplace(im, s)
        e = hessian_matrix_eigvals(hessian_matrix(im, sigma=s, order='rc',
                                                  use_gaussian_derivatives=True))
        yield f'hess{s:g}_1', e[0]
        yield f'hess{s:g}_2', e[1]

SIGMAS = spec['features']['globals']['sigmas']
names = [n for n, _ in feature_stack(np.zeros((8, 8), np.uint16), SIGMAS)]
print(len(names), 'features:', ', '.join(names[:8]), '...')

In [ ]:
crop = montage[y0:y0+300, x0:x0+300]
show = ['identity', 'gauss2', 'dog2', 'gradmag2', 'log2', 'hess2_2']
planes = dict(feature_stack(crop, SIGMAS))

fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
for a, k in zip(axes, show):
    p = planes[k]
    lim = np.percentile(np.abs(p), 99)
    a.imshow(p, cmap='gray' if k in ('identity', 'gauss2') else 'RdBu_r',
             vmin=None if k in ('identity','gauss2') else -lim,
             vmax=np.percentile(p, 99.5) if k in ('identity','gauss2') else lim)
    a.set_title(k, fontsize=9); a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout()
plt.suptitle('What the classifier actually sees, at sigma 2', x=0.02, y=1.10,
             ha='left', fontsize=12, weight='semibold')
plt.show()

---
## 5. Training the forest

Sample the feature stack at the scribbled pixels, and fit. Weka's
`FastRandomForest` defaults to 200 trees, so scikit-learn is given the same.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def sample_at(img, coords, sigmas):
    r = coords[:, 1], coords[:, 0]        # Labkit writes [x, y]
    return np.stack([p[r] for _, p in feature_stack(img, sigmas)], axis=1)

X, yv = [], []
for cls_i, name in enumerate(spec['classNames']):        # background=0, Mitochondria=1
    key = next(k for k in scribbles if k.lower() == name.lower())
    f = sample_at(montage, scribbles[key], SIGMAS)
    X.append(f); yv.append(np.full(len(f), cls_i))
X = np.concatenate(X); yv = np.concatenate(yv)
print(f'{X.shape[0]:,} training pixels x {X.shape[1]} features')

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=20260905).fit(X, yv)
print('trained')

In [ ]:
imp = sorted(zip(names, rf.feature_importances_), key=lambda t: -t[1])[:10]
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.barh([k for k, _ in imp][::-1], [v for _, v in imp][::-1],
        color=C['blue'], height=0.68)
ax.set_xlabel('importance'); ax.set_title('Which filters the forest leans on')
ax.set_axisbelow(True); ax.grid(axis='x', alpha=0.9); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

---
## 6. Does it agree with Labkit?

The honest test. Segment images with both and measure intersection over union.

**The benchmark matters more than the number.** Classifiers v4 and v5 were made
by the same person, from the same images, with the same tool, differing only in
where the scribbles went - and they agree on **67%** of segmented pixels. That is
the noise floor of human annotation, and it is the bar the port has to clear.

In [ ]:
def segment(img, rf, sigmas):
    planes = [p.ravel() for _, p in feature_stack(img, sigmas)]
    return rf.predict(np.stack(planes, axis=1)).reshape(img.shape).astype(bool)

def iou(a, b):
    u = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / u if u else 1.0

ids = sorted(f[:-len('.ome.tif')] for f in os.listdir('port_example/images'))
res = []
for i in ids:
    img = tifffile.imread(f'port_example/images/{i}.ome.tif')
    ref = tifffile.imread(f'port_example/masks_labkit/{i}.ome_segmentation.tif').astype(bool)
    got = segment(img, rf, SIGMAS)
    res.append((i, iou(got, ref), ref.mean(), got.mean(), img, ref, got))
    print(f'  {i}  IoU {res[-1][1]:.3f}')

v = np.array([r[1] for r in res])
print(f'\nmedian IoU {np.median(v):.3f}   min {v.min():.3f}   all above 0.67: {bool((v>0.67).all())}')

In [ ]:
i, io_, fl, fp, img, ref, got = max(res, key=lambda r: -abs(r[1] - np.median(v)))
y0c, x0c = [s // 2 - 150 for s in img.shape[:2]]
sl = (slice(y0c, y0c+300), slice(x0c, x0c+300))

fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
ax[0].imshow(img[sl], cmap='gray', vmax=np.percentile(img[sl], 99.5)); ax[0].set_title('image')
ax[1].imshow(ref[sl], cmap='gray'); ax[1].set_title('Labkit (Java)')
ax[2].imshow(got[sl], cmap='gray'); ax[2].set_title('this notebook (Python)')

rgb = np.zeros(ref[sl].shape + (3,))
rgb[..., 0] = ref[sl] & ~got[sl]      # Labkit only
rgb[..., 1] = ref[sl] & got[sl]       # both
rgb[..., 2] = got[sl] & ~ref[sl]      # Python only
ax[3].imshow(rgb); ax[3].set_title('green agree | red Labkit | blue Python')
for a in ax:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout()
plt.suptitle(f'{i}   IoU {io_:.3f}', x=0.02, y=1.06, ha='left',
             fontsize=12, weight='semibold')
plt.show()

Disagreement sits on object boundaries - a pixel either side of an edge - not on
whole structures found by one and missed by the other. That is the benign kind.

---
## 7. Does the *index* survive the swap?

Pixel agreement is not the deliverable. The index is. A port could agree on 93%
of pixels and still shift the score if the disagreement were systematic.

Measured over 40 images, outside this notebook for time:

| | |
|---|---|
| IoU against Labkit v5 | median **0.931**, IQR 0.921-0.941, min 0.749 |
| every image above the 0.67 annotation benchmark | **yes** |
| foreground fraction | Labkit 0.0759, Python 0.0766 |
| **index correlation** | **r = 0.990** |
| absolute index difference | median 0.027, p90 0.062, max 0.167 |
| mixed-band assignment agreement | 92% |

Reproduce with `scripts/labkit_port.py --train --compare 40`.

A median index shift of 0.027 on a 0-to-1 scale is smaller than the width of the
mixed band, and far smaller than the difference between the calls it
distinguishes.

---
## 8. Where that leaves the port

| | Status |
|---|---|
| Pixel classification | **ported and validated** - no Fiji, no GUI |
| Feature extraction, model, index, QC gate | already Python |
| Montage, projection, channel choice | trivial numpy |
| Reading `.oir` | Bio-Formats, headless - Java present but invisible |
| Reading `.lif` | `readlif`, pure Python |
| Drawing *new* scribbles | napari - **training only, not needed to use the tool** |

The usage path needs none of the GUI steps. A user of the finished tool would run
one command; the only Java left is a library reading a vendor file format, which
is ordinary in this field and invisible to them.

### What is not settled

- The DoG sigma ratio is a guess, and the 0.990 correlation is what justifies it.
  Reading the Labkit source would settle it and might close the remaining gap.
- 40 images, one classifier, one microscope. The Leica set tests it properly.
- Retraining still needs someone to draw scribbles. napari can host that, but
  moving the annotation interface changes what an annotator sees, and this
  project has already measured how much annotation alone moves the answer: a
  third of segmented pixels between v4 and v5. Swap that interface deliberately,
  and re-measure when you do.